In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import mlflow
import mlflow.catboost
from catboost import CatBoostClassifier
import psycopg
from autofeat import AutoFeatClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss


TABLE_NAME = 'clean_users_churn' # таблица с данными

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

In [2]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

In [3]:
df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2133,3023-GFLBR,2017-03-01,2019-12-01,Month-to-month,No,Credit card (automatic),86.15,2745.7,Fiber optic,...,No,No,No,Yes,Female,0,Yes,Yes,Yes,1
1,837,0727-BMPLR,2015-04-01,2019-11-01,One year,Yes,Electronic check,100.00,5509.3,Fiber optic,...,Yes,No,Yes,Yes,Female,1,No,No,Yes,1


In [19]:
exclude_columns = ['id', 'customer_id', 'target', 'begin_date', 'end_date']
features_no_target = [col for col in df.columns if col not in exclude_columns]
target = ['target']

split_column = "begin_date"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features_no_target],
    df[target],
    test_size=test_size,
    shuffle=False,
    random_state=42
)

cat_features = [
    'paperless_billing',
    'payment_method',
    'internet_service',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'gender',
    'senior_citizen',
    'partner',
    'dependents',
    'multiple_lines',
    'type'
]
num_features = ["monthly_charges", "total_charges"]

features = cat_features + num_features

transformations = ('1/', 'log', 'abs', 'sqrt')

afc = AutoFeatClassifier(categorical_cols=cat_features,
                         feateng_cols=num_features,
                         transformations=transformations,
                         feateng_steps=3,
                         n_jobs=-1,
                         verbose=1, 
                         featsel_runs=0)

X_train_features = afc.fit_transform(X_train, y_train)
X_test_features = afc.transform(X_test)

/home/mle-user/mle-mlflow/.venv_mle_mlflow/lib/python3.10/site-packages/sklearn/utils/validation.py:1183: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
2024-11-10 08:12:02,938 INFO: [AutoFeat] The 3 step feature engineering process could generate up to 372 features.
2024-11-10 08:12:02,939 INFO: [AutoFeat] With 5615 data points this new feature matrix would use about 0.01 gb of space.


2024-11-10 08:12:02,943 INFO: [feateng] Step 1: transformation of original features


2024-11-10 08:12:03,832 INFO: [feateng] Generated 6 transformed features from 2 original features - done.
2024-11-10 08:12:03,835 INFO: [feateng] Step 2: first combination of features
2024-11-10 08:12:04,205 INFO: [feateng] Generated 110 feature combinations from 28 original feature tuples - done.
2024-11-10 08:12:04,209 INFO: [feateng] Step 3: transformation of new features


2024-11-10 08:12:04,587 INFO: [feateng] Generated 273 transformed features from 110 original features - done.
2024-11-10 08:12:04,597 INFO: [feateng] Generated altogether 409 new features in 3 steps
2024-11-10 08:12:04,600 INFO: [feateng] Removing correlated features, as well as additions at the highest level
2024-11-10 08:12:04,725 INFO: [feateng] Generated a total of 118 additional features
2024-11-10 08:12:04,730 WARNING: [AutoFeat] Not performing feature selection.
2024-11-10 08:12:04,738 INFO: [AutoFeat] Computing 69 new features.


2024-11-10 08:12:20,767 INFO: [AutoFeat]    69/   69 new features ...done.
2024-11-10 08:12:20,773 INFO: [AutoFeat] Final dataframe with 104 feature columns (87 new).
2024-11-10 08:12:20,774 INFO: [AutoFeat] Training final classification model.
2024-11-10 08:12:25,584 INFO: [AutoFeat] Trained model: largest coefficients:
2024-11-10 08:12:25,585 INFO: [-0.00695376]
2024-11-10 08:12:25,586 INFO: 0.629847 * cat_type_Month-to-month
2024-11-10 08:12:25,588 INFO: 0.565873 * cat_type_Two year
2024-11-10 08:12:25,589 INFO: 0.408658 * monthly_charges/sqrt(total_charges)
2024-11-10 08:12:25,591 INFO: 0.369035 * cat_payment_method_Electronic check
2024-11-10 08:12:25,592 INFO: 0.230377 * cat_tech_support_No
2024-11-10 08:12:25,593 INFO: 0.221629 * cat_tech_support_Yes
2024-11-10 08:12:25,594 INFO: 0.220531 * log(monthly_charges/total_charges)
2024-11-10 08:12:25,596 INFO: 0.215256 * cat_online_security_No
2024-11-10 08:12:25,597 INFO: 0.206509 * cat_online_security_Yes
2024-11-10 08:12:25,598 INF

In [23]:
X_test_features.head()

,monthly_charges,total_charges,cat_paperless_billing_No,cat_paperless_billing_Yes,cat_payment_method_Bank transfer (automatic),cat_payment_method_Credit card (automatic),cat_payment_method_Electronic check,cat_payment_method_Mailed check,cat_internet_service_DSL,cat_internet_service_Fiber optic,...,log(sqrt(monthly_charges)*sqrt(total_charges)),1/(sqrt(monthly_charges) - sqrt(total_charges)),1/(-sqrt(monthly_charges) + sqrt(total_charges)),1/(sqrt(total_charges) + 1/total_charges),1/(-sqrt(total_charges) + 1/total_charges),1/(sqrt(total_charges) - 1/total_charges),1/(sqrt(total_charges) + log(total_charges)),1/(sqrt(total_charges)*log(total_charges)),1/(-sqrt(total_charges) + log(total_charges)),1/(sqrt(total_charges) - log(total_charges))
0,69.95,330.15,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,5.023664,-0.101974,0.101974,0.055027,-0.055045,0.055045,0.041720,0.009490,-0.080838,0.080838
1,19.4,168.65,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,4.046549,-0.116523,0.116523,0.076968,-0.077038,0.077038,0.055205,0.015017,-0.127247,0.127247
2,79.15,317.25,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,5.065517,-0.112172,0.112172,0.056134,-0.056153,0.056153,0.042425,0.009748,-0.082975,0.082975
3,20.25,174.7,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,4.085613,-0.114713,0.114713,0.075625,-0.075691,0.075691,0.054406,0.014654,-0.124157,0.124157
4,91.85,257.05,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,5.034714,-0.155064,0.155064,0.062357,-0.062387,0.062387,0.046335,0.011240,-0.095388,0.095388


In [24]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = 'https://storage.yandexcloud.net'
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

artifact_path = "afc"
experiment_id = mlflow.get_experiment_by_name("churn_laptev_ilya_sergeevich_2").experiment_id

with mlflow.start_run(run_name="autofeat", experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    afc_info = mlflow.sklearn.log_model(afc, artifact_path=artifact_path)

2024-11-10 08:16:27,451 INFO: Found credentials in environment variables.


In [29]:
pd.set_option('display.max_rows', None)

In [38]:
X_train_features['monthly_charges'] = X_train_features['monthly_charges'].astype('float')
X_train_features['total_charges'] = X_train_features['total_charges'].astype('float')

X_test_features['monthly_charges'] = X_test_features['monthly_charges'].astype('float')
X_test_features['total_charges'] = X_test_features['total_charges'].astype('float')

In [47]:
model = CatBoostClassifier(iterations=1000, learning_rate=0.01, verbose=0)

In [48]:
model.fit(X_train_features, y_train)

In [51]:
# Предсказанные значения и вероятности
proba = model.predict_proba(X_test_features)[:, 1]
prediction = model.predict(X_test_features)

# Истинные метки и данные для предсказания
y_true = y_test

# Заведите словарь со всеми метриками
metrics = {}

# Посчитайте метрики из модуля sklearn.metrics с нормализацией
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()

auc = roc_auc_score(y_true, proba)
precision = precision_score(y_true, prediction)
recall = recall_score(y_true, prediction)
f1 = f1_score(y_true, prediction)
logloss = log_loss(y_true, proba)

# Запишите значения метрик в словарь 
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [52]:
metrics

{'err1': 0.4658119658119658,
 'err2': 0.47507122507122507,
 'auc': 0.7346475527206457,
 'precision': 0.5049205147615443,
 'recall': 0.9985029940119761,
 'f1': 0.6706887883358471,
 'logloss': 1.322946922021993}

In [53]:
EXPERIMENT_NAME = 'churn_laptev_ilya_sergeevich_2'
RUN_NAME = 'cb_autofeat'
REGISTRY_MODEL_NAME = 'cb_with_autofeat'

pip_requirements = '../requirements.txt'
signature = mlflow.models.infer_signature(X_test_features, prediction)
input_example = X_test_features[:10]
metadata = {'model_type': 'monthly'}

In [54]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_metrics(metrics)
    # ваш код здесь
    model_info = mlflow.catboost.log_model(cb_model=model,
                                          metadata=metadata,
                                          artifact_path='models',
                                          signature=signature,
                                          pip_requirements=pip_requirements,
                                          input_example=input_example,
                                          registered_model_name=REGISTRY_MODEL_NAME,
                                          await_registration_for=60)

2024/11/10 08:34:42 WARNING mlflow.models.model: Logging model metadata to the tracking server has failed. The model artifacts have been logged successfully under s3://s3-student-mle-20240920-1460ff9140/4/cdb6047e3817441da3fc192dec7f590c/artifacts. Set logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)` to see the full traceback.
Successfully registered model 'cb_with_autofeat'.
2024/11/10 08:34:42 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: cb_with_autofeat, version 1
Created version '1' of model 'cb_with_autofeat'.
